# Popularity

Run `01_data` first to generate the ratings and the fixed user sample. The baseline counts
all ratings in the fitting window and recommends up to 10 unseen films per sampled user.
A film is eligible once it reaches `MIN_RATINGS` in that same window. The default of 10
follows the data pipeline. Ties in rating count are broken by ascending `movieId`.

Keep `EVALUATION_SPLIT = "val"` for development. Set it to `"test"` for the final run.

| Evaluation split | Fitting window | Output in `paths.results_dir()` |
| --- | --- | --- |
| `val` | `train` | `popularity_val.parquet` |
| `test` | `train` + `val` | `popularity_test.parquet` |

The fitting window determines both the scores and which movies count as already rated.
In Colab the outputs are written under `MyDrive/movierec/results`.

In [ ]:

import importlib
import os
import subprocess
import sys
import types

try:
    from google.colab import drive
except ModuleNotFoundError:
    drive = None

if drive is not None:
    from IPython import get_ipython

    shell = get_ipython()
    BRANCH = "main"
    REPO_URL = "https://github.com/Cxsee/AI-in-Industry-project.git"
    EXTRAS = ""
    DRIVE_DIR = "/content/drive/MyDrive/movierec"
    REPO_DIR = "/content/AI-in-Industry-project"

    drive.mount("/content/drive")
    os.environ["MOVIEREC_DATA_DIR"] = f"{DRIVE_DIR}/data"
    os.environ["MOVIEREC_RESULTS_DIR"] = f"{DRIVE_DIR}/results"
    os.makedirs(os.environ["MOVIEREC_DATA_DIR"], exist_ok=True)
    os.makedirs(os.environ["MOVIEREC_RESULTS_DIR"], exist_ok=True)

    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)

    shell.run_line_magic("cd", REPO_DIR)
    shell.system("git pull -q")
    target = f".[{EXTRAS}]" if EXTRAS else "."
    shell.system('pip install -q -e "{target}"')

    # The running kernel doesn't pick up an editable install made after it started.
    if f"{REPO_DIR}/src" not in sys.path:
        sys.path.insert(0, f"{REPO_DIR}/src")

    # Colab's IPython 7.34 still imports `imp`, which Python 3.13 removed.
    sys.modules.setdefault("imp", types.SimpleNamespace(reload=importlib.reload))
    shell.run_line_magic("load_ext", "autoreload")
    shell.run_line_magic("autoreload", "2")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/AI-in-Industry-project
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for movierec (pyproject.toml) ... done


In [4]:
from itertools import islice
from pathlib import Path
from tempfile import TemporaryDirectory

import polars as pl

from movierec import paths

EVALUATION_SPLIT = "val"
TOP_K = 10
MIN_RATINGS = 10

FIT_SPLITS = {"val": ("train",), "test": ("train", "val")}

## Count ratings and collect user histories

Count ratings from all users in the fitting window. Only the sampled users need an
individual history for filtering the ranked catalog.

In [5]:
if EVALUATION_SPLIT not in FIT_SPLITS:
    raise ValueError('EVALUATION_SPLIT must be "val" or "test".')
if TOP_K < 1 or MIN_RATINGS < 1:
    raise ValueError("TOP_K and MIN_RATINGS must be positive.")

users = pl.read_parquet(paths.data_dir() / "users_sample.parquet").sort("sample_rank")
user_ids = users.select("userId")
history = (
    pl.scan_parquet(paths.data_dir() / "ratings.parquet")
    .filter(pl.col("split").is_in(FIT_SPLITS[EVALUATION_SPLIT]))
    .select("userId", "movieId")
)

popularity, sampled_histories = pl.collect_all(
    [
        history.group_by("movieId")
        .len(name="score")
        .filter(pl.col("score") >= MIN_RATINGS)
        .with_columns(pl.col("score").cast(pl.Int64))
        .sort(["score", "movieId"], descending=[True, False]),
        history.join(user_ids.lazy(), on="userId", how="semi")
        .group_by("userId")
        .agg(pl.col("movieId")),
    ]
)
print(f"{popularity.height:,} eligible films for {EVALUATION_SPLIT} evaluation.")

22,280 eligible films for val evaluation.


## Recommend unseen films and save

Each row contains `userId`, `movieId`, one-based `rank` and `score` (the film's rating count).
Users get shorter lists if the eligible catalog contains fewer than 10 unseen films.
Read each user's movies in rank order when passing them to the evaluation metrics.

In [6]:
seen_by_user = {user_id: set(movie_ids) for user_id, movie_ids in sampled_histories.iter_rows()}
ranked_movies = popularity.rows()
rows = []
full_rankings = 0

for user_id in user_ids["userId"]:
    seen = seen_by_user.get(user_id, ())
    unseen = ((movie_id, score) for movie_id, score in ranked_movies if movie_id not in seen)
    top = list(islice(unseen, TOP_K))
    full_rankings += len(top) == TOP_K
    rows.extend((user_id, movie_id, rank, score) for rank, (movie_id, score) in enumerate(top, 1))

recommendations = pl.DataFrame(
    rows,
    schema={
        "userId": user_ids.schema["userId"],
        "movieId": popularity.schema["movieId"],
        "rank": pl.Int64,
        "score": pl.Int64,
    },
    orient="row",
)

output = paths.results_dir() / f"popularity_{EVALUATION_SPLIT}.parquet"
output.parent.mkdir(parents=True, exist_ok=True)
with TemporaryDirectory(dir=output.parent) as temp_dir:
    temporary_output = Path(temp_dir) / output.name
    recommendations.write_parquet(temporary_output)
    temporary_output.replace(output)

print(f"Saved {recommendations.height:,} recommendations to {output}")
print(f"Users with {TOP_K} recommendations: {full_rankings:,}/{users.height:,}")
recommendations.head(TOP_K)

Saved 20,000 recommendations to /content/drive/MyDrive/movierec/results/popularity_val.parquet
Users with 10 recommendations: 2,000/2,000


userId,movieId,rank,score
i32,i32,i64,i64
49167,296,1,80491
49167,356,2,80084
49167,593,3,73450
49167,260,4,69757
49167,2959,5,62565
49167,4993,6,59536
49167,527,7,58836
49167,480,8,57706
49167,7153,9,55590
